# CAAR-CDSS: Kaggle T4 Benchmark Evaluation
**Confidence-Aware Adaptive Retrieval Clinical Decision Support System**

This notebook runs the full MedQA evaluation benchmark on Kaggle using a free NVIDIA T4 GPU (16 GB VRAM).
- **Generation**: `meta-llama/Llama-3.1-8B-Instruct` (fp16 unquantized)
- **RAGAS Judge**: `KagglePipelineJudge` (Reuses loaded fp16 pipeline — **0 API calls**)
- **Verification**: `DeBERTa-v3-large` via Hugging Face Inference API (0 local VRAM)

### Resource Management
This notebook includes aggressive disk/GPU cleanup between steps to stay within Kaggle limits (~20 GB disk, 16 GB VRAM).

In [ ]:
# 1. Setup Secrets, Environment, and Disk Monitoring
import os
import shutil
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    print("✅ HF_TOKEN configured successfully.")
except Exception as e:
    print(f"⚠️ Could not fetch HF_TOKEN from Kaggle Secrets: {e}")
    print("Please add 'HF_TOKEN' in Kaggle -> Add-ons -> Secrets.")

# Database URL
os.environ["DATABASE_URL"] = "sqlite+aiosqlite:////kaggle/working/caar_cdss.db"

# HF cache — use /kaggle/working so we can manage disk
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.makedirs("/kaggle/working/results", exist_ok=True)

# --- Disk dashboard ---
usage = shutil.disk_usage("/kaggle/working")
free_gb = usage.free / (1024**3)
total_gb = usage.total / (1024**3)
used_gb = usage.used / (1024**3)
print(f"\n💾 Disk: {free_gb:.1f} GB free / {total_gb:.1f} GB total ({used_gb:.1f} GB used)")
if free_gb < 5.0:
    print("🛑 WARNING: Low disk space! Consider using Option C (pre-built Chroma) in step 5.")
elif free_gb < 15.0:
    print("⚠️  Disk space is moderate. Use --limit 2000 for ingestion.")
else:
    print("✅ Disk space OK for full run.")

In [ ]:
# 2. Install Project Dependencies (trimmed to evaluation-only packages)
!pip install -q --upgrade pip
!pip install -q transformers accelerate bitsandbytes sentence-transformers datasets chromadb FlagEmbedding ragas
!pip install -q rank-bm25 pydantic pydantic-settings matplotlib pandas

In [ ]:
# 3. Clone / Setup Repository Files
import shutil
from pathlib import Path

# If using Kaggle dataset input
repo_input = Path("/kaggle/input/caar-cdss-repo")
if repo_input.exists():
    !cp -r /kaggle/input/caar-cdss-repo/* /kaggle/working/
    print("✅ Copied repo files from Kaggle input dataset.")
else:
    print("ℹ️ Running directly in working directory. Ensure src/ and configs/ are present.")

In [ ]:
# 4. Check GPU Hardware & Disk Status
!python -c "from src.config import detect_hardware; import json; hw = detect_hardware(); print(json.dumps(hw, indent=2))"

In [ ]:
# 5. Ingest Guidelines Corpus (reduced limit for disk safety)
# Option A: Stream & index 2,000 articles (~5-8 min, ~2 GB disk)
!python -m src.cli ingest --corpus epfl-llm/guidelines --limit 2000 --chroma-dir /kaggle/working/chroma_db

# Option B: 5,000 articles (uncomment if disk > 15 GB free)
# !python -m src.cli ingest --corpus epfl-llm/guidelines --limit 5000 --chroma-dir /kaggle/working/chroma_db

# Option C: Mount pre-built Chroma DB as Kaggle Dataset (fastest, 0 extra disk)
# !cp -r /kaggle/input/caar-cdss-chroma/* /kaggle/working/chroma_db/

# --- Cleanup unused HF cache entries after embedding model download ---
!python -c "
from src.utils.disk_utils import cleanup_hf_cache, check_disk
cleanup_hf_cache(keep_models=['BAAI/bge-m3', 'BAAI/bge-small-en-v1.5'])
check_disk('/kaggle/working')
"

In [ ]:
# 6. Run MedQA Evaluation (N=50, with GC cleanup per query)
!python -m src.experiments.ragas_eval \
    --benchmark medqa \
    --n 50 \
    --mode kaggle_fp16 \
    --judge-model meta-llama/Llama-3.1-8B-Instruct \
    --max-workers 1 \
    --gc-after-each \
    --output /kaggle/working/results/medqa_50_ragas.json

# --- Post-step cleanup ---
!python -c "from src.utils.disk_utils import cleanup_after_step; cleanup_after_step('ragas-eval')"

In [ ]:
# 7. View RAGAS Results
import json
from pathlib import Path

results_file = Path("/kaggle/working/results/medqa_50_ragas.json")
if results_file.exists():
    results = json.loads(results_file.read_text())
    print("=== MedQA RAGAS Evaluation Results ===")
    for metric, score in results.items():
        print(f"  {metric}: {score:.4f}")
else:
    print("⚠️ RAGAS results file not found. Check cell 6 output for errors.")

In [ ]:
# 8. Run Comparative Evaluation (Vanilla vs Hybrid vs AEB) on MedQA + Seeds
!python -m src.experiments.runner \
    --benchmarks medqa seeds \
    --n 50 \
    --mode kaggle_fp16 \
    --gc-after-each \
    --output /kaggle/working/results/comparative

# --- Post-step cleanup ---
!python -c "from src.utils.disk_utils import cleanup_after_step; cleanup_after_step('runner-comparative')"

In [ ]:
# 9. (Optional) Threshold Sweep — only run if enough time remains
# Estimated time: ~30-60 min for 10 thresholds × seeds
import shutil
free_gb = shutil.disk_usage('/kaggle/working').free / (1024**3)
print(f"💾 Free disk: {free_gb:.1f} GB")

if free_gb > 3.0:
    !python -m src.experiments.runner \
        --threshold-sweep \
        --benchmarks seeds \
        --n 8 \
        --mode kaggle_fp16 \
        --gc-after-each \
        --output /kaggle/working/results/threshold_sweep
else:
    print("⚠️ Skipping threshold sweep — insufficient disk space.")

In [ ]:
# 10. (Optional) Embedding Ablation — only run if enough time remains
import shutil
free_gb = shutil.disk_usage('/kaggle/working').free / (1024**3)
print(f"💾 Free disk: {free_gb:.1f} GB")

if free_gb > 3.0:
    !python -m src.experiments.runner \
        --embedding-ablation \
        --benchmarks seeds \
        --n 8 \
        --mode kaggle_fp16 \
        --gc-after-each \
        --output /kaggle/working/results/embedding_ablation
else:
    print("⚠️ Skipping embedding ablation — insufficient disk space.")

In [ ]:
# 11. View All Results
import json
from pathlib import Path

results_dir = Path("/kaggle/working/results")
for f in sorted(results_dir.rglob("*.json")):
    print(f"\n=== {f.relative_to(results_dir)} ===")
    try:
        with open(f) as fp:
            data = json.load(fp)
            if isinstance(data, dict) and "method" in data:
                print(f"  Method: {data['method']}, Acc: {data.get('accuracy', 0):.2f}, Abstain: {data.get('abstention_rate', 0):.2f}, Halluc: {data.get('avg_hallucination', 0):.2f}, AvgK: {data.get('avg_retrieval_k', 0):.1f}")
            elif isinstance(data, dict):
                for k, v in data.items():
                    if isinstance(v, dict) and "accuracy" in v:
                        print(f"  {k}: Acc={v['accuracy']:.2f}, Abstain={v.get('abstention_rate', 0):.2f}")
                    elif isinstance(v, (int, float)):
                        print(f"  {k}: {v}")
    except Exception as e:
        print(f"  Error reading: {e}")

In [ ]:
# 12. Package Results for Download
import zipfile
import os
import shutil
from IPython.display import FileLink

# Check disk before zipping
free_gb = shutil.disk_usage('/kaggle/working').free / (1024**3)
print(f"💾 Free disk: {free_gb:.1f} GB")

RESULTS_DIR = Path("/kaggle/working/results")
zip_path = "/kaggle/working/results.zip"

if free_gb < 0.5:
    print("🛑 Not enough disk to create zip. Download individual JSON files from /kaggle/working/results/")
else:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(str(RESULTS_DIR)):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, str(RESULTS_DIR))
                zipf.write(file_path, arcname)
    
    print("🎉 Completed successfully! Click below to download results:")
    FileLink(r"results.zip")